# KOSPI200 지수옵션 실시간 호가 수집기
키움 Open API를 사용하여 KOSPI200 지수옵션 호가/시세(그릭스, IV 포함) 및 KOSPI200 지수 데이터를 CSV로 저장

## 1. 설정

In [1]:
# === 사용자 설정 ===
OPTION_CODE = "B0164850"  # KOSPI200 지수옵션 (코스피200 C 202604 850.0) - ATM
INDEX_CODE = "201"        # KOSDAQ 종합지수 (업종코드)
SCREEN_NO = "1000"        # 화면번호

## 2. 라이브러리 Import

In [2]:
import sys
import os
from datetime import datetime
import pandas as pd
from PyQt5.QtWidgets import QApplication
from PyQt5.QAxContainer import QAxWidget
from PyQt5.QtCore import QEventLoop, QTimer

## 3. CSV 저장 설정 및 FID 매핑

In [3]:
# 저장 디렉토리 설정
DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

# CSV 파일 경로 생성
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M")
CSV_FILENAME = f"kospi200_option_{OPTION_CODE}_{timestamp_str}.csv"
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

# CSV 컬럼 정의 (한글)
CSV_COLUMNS = [
    # 기본 정보
    "시간", "데이터유형",

    # 옵션 시세
    "옵션코드", "옵션체결시간", "옵션현재가", "옵션거래량",
    "옵션이론가",
    "옵션IV",
    "옵션델타",
    "옵션감마",
    "옵션세타",
    "옵션베가",
    "옵션로",
    "옵션미결제약정",

    # 옵션 호가
    "옵션매도호가1", "옵션매도잔량1",
    "옵션매도호가2", "옵션매도잔량2",
    "옵션매도호가3", "옵션매도잔량3",
    "옵션매도호가4", "옵션매도잔량4",
    "옵션매도호가5", "옵션매도잔량5",
    "옵션매수호가1", "옵션매수잔량1",
    "옵션매수호가2", "옵션매수잔량2",
    "옵션매수호가3", "옵션매수잔량3",
    "옵션매수호가4", "옵션매수잔량4",
    "옵션매수호가5", "옵션매수잔량5",
    "옵션총매도잔량", "옵션총매수잔량",

    # KOSPI200 지수
    "지수코드", "지수체결시간", "지수현재가",
    "지수전일대비", "지수등락률",
    "지수거래량", "지수시가"
]

# FID 매핑 - 옵션이론가 (그릭스 데이터)
FID_OPTION_GREEKS = {
    "옵션이론가": 182,
    "옵션IV": 190,
    "옵션델타": 191,
    "옵션감마": 192,
    "옵션세타": 193,
    "옵션베가": 194,
    "옵션미결제약정": 195,
}

# FID 매핑 - 옵션시세 (체결 데이터)
FID_OPTION_TRADE = {
    "옵션체결시간": 20,
    "옵션현재가": 10,
    "옵션거래량": 15,
}

# FID 매핑 - 옵션호가잔량
FID_OPTION_QUOTE = {
    # 매도호가
    "옵션매도호가1": 41, "옵션매도호가2": 42, "옵션매도호가3": 43, "옵션매도호가4": 44, "옵션매도호가5": 45,
    # 매도잔량
    "옵션매도잔량1": 61, "옵션매도잔량2": 62, "옵션매도잔량3": 63, "옵션매도잔량4": 64, "옵션매도잔량5": 65,
    # 매수호가
    "옵션매수호가1": 51, "옵션매수호가2": 52, "옵션매수호가3": 53, "옵션매수호가4": 54, "옵션매수호가5": 55,
    # 매수잔량
    "옵션매수잔량1": 71, "옵션매수잔량2": 72, "옵션매수잔량3": 73, "옵션매수잔량4": 74, "옵션매수잔량5": 75,
    # 총잔량
    "옵션총매도잔량": 121,
    "옵션총매수잔량": 125,
}

# FID 매핑 - 업종지수 (KOSPI200 지수)
FID_INDEX = {
    "지수체결시간": 20,
    "지수현재가": 10,
    "지수전일대비": 11,
    "지수등락률": 12,
    "지수거래량": 15,
    "지수시가": 16,
}

# 전체 FID 통합 (실시간 등록용)
ALL_OPTION_FIDS = set(FID_OPTION_GREEKS.values()) | set(FID_OPTION_TRADE.values()) | set(FID_OPTION_QUOTE.values())
ALL_INDEX_FIDS = set(FID_INDEX.values())

## 4. KiwoomIndexOptionCollector 클래스 정의

In [4]:
class KiwoomIndexOptionCollector:
    def __init__(self, option_code, index_code, screen_no, csv_path, interval_ms=1000):
        self.option_code = option_code
        self.index_code = index_code
        self.screen_no = screen_no
        self.csv_path = csv_path
        self.interval_ms = interval_ms

        self.app = QApplication(sys.argv)
        self.kiwoom = QAxWidget("KHOPENAPI.KHOpenAPICtrl.1")

        self.kiwoom.OnEventConnect.connect(self.on_event_connect)
        self.kiwoom.OnReceiveRealData.connect(self.on_receive_real_data)

        self.login_event_loop = QEventLoop()
        self.connected = False
        self.data_count = 0

        # 마지막 수신 데이터 캐시
        self.last_data = {col: "" for col in CSV_COLUMNS}
        self.last_data["옵션코드"] = option_code
        self.last_data["지수코드"] = index_code

        # 데이터 수신 여부 플래그
        self.data_received = False

        # 수신된 모든 real_type 기록 (디버깅용)
        self.received_types = set()

        # 1초 타이머 설정
        self.timer = QTimer()
        self.timer.timeout.connect(self.on_timer)

        # CSV 헤더 작성
        self._init_csv()

    def _init_csv(self):
        """CSV 파일 초기화 (헤더 작성)"""
        df = pd.DataFrame(columns=CSV_COLUMNS)
        df.to_csv(self.csv_path, index=False, encoding="utf-8-sig")
        print(f"CSV 파일 생성: {self.csv_path}")

    def login(self):
        """로그인"""
        self.kiwoom.dynamicCall("CommConnect()")
        self.login_event_loop.exec_()

    def on_event_connect(self, err_code):
        """로그인 결과 처리"""
        if err_code == 0:
            print("\n[성공] 로그인 완료")
            self.connected = True

            user_name = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "USER_NAME")
            server_type = self.kiwoom.dynamicCall("GetLoginInfo(QString)", "GetServerGubun")

            print(f"사용자: {user_name}")
            print(f"서버: {'모의투자' if server_type == '1' else '실투자'}")
        else:
            print(f"\n[실패] 로그인 실패 (에러코드: {err_code})")
            self.connected = False

        self.login_event_loop.exit()

    def set_real_reg(self):
        """실시간 등록 (옵션 + KOSPI200 지수)"""
        option_fid_list = ";".join([str(fid) for fid in ALL_OPTION_FIDS])
        index_fid_list = ";".join([str(fid) for fid in ALL_INDEX_FIDS])

        # 옵션 종목 등록
        result1 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.option_code, option_fid_list, "0"
        )
        print(f"옵션 실시간 등록 ({self.option_code}): {result1}")

        # KOSPI200 지수 추가 등록
        result2 = self.kiwoom.dynamicCall(
            "SetRealReg(QString, QString, QString, QString)",
            self.screen_no, self.index_code, index_fid_list, "1"
        )
        print(f"지수 실시간 등록 ({self.index_code}): {result2}")

        return result1, result2

    def get_real_data(self, code, fid):
        """실시간 데이터 조회"""
        value = self.kiwoom.dynamicCall(
            "GetCommRealData(QString, int)", code, fid
        ).strip()
        return value

    def _clean_value(self, value):
        """값 정리 (부호 제거)"""
        if value and (value.startswith("+") or value.startswith("-")):
            if value[1:].replace(".", "").isdigit():
                return value[1:]
        return value

    def on_receive_real_data(self, code, real_type, real_data):
        """실시간 데이터 수신 → 캐시 업데이트"""

        # [디버깅] 새로운 real_type 출력
        if real_type not in self.received_types:
            self.received_types.add(real_type)
            print(f"\n>>> [NEW TYPE] real_type = '{real_type}' (code: {code}) <<<")

        # 옵션이론가 처리 (그릭스)
        if real_type == "옵션이론가" and code == self.option_code:
            for key, fid in FID_OPTION_GREEKS.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 옵션시세 처리 (체결)
        elif real_type in ("옵션시세", "주식옵션시세") and code == self.option_code:
            for key, fid in FID_OPTION_TRADE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 옵션호가잔량 처리
        elif real_type in ("옵션호가잔량", "주식옵션호가잔량") and code == self.option_code:
            for key, fid in FID_OPTION_QUOTE.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

        # 업종지수 처리 (KOSPI200)
        elif real_type in ("업종지수", "업종등락") and code == self.index_code:
            for key, fid in FID_INDEX.items():
                value = self._clean_value(self.get_real_data(code, fid))
                if value:
                    self.last_data[key] = value
            self.data_received = True

    def on_timer(self):
        """1초마다 현재 캐시 상태를 CSV에 저장"""
        if not self.data_received:
            return

        # 타임스탬프 업데이트 (1초 단위)
        self.last_data["시간"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.last_data["데이터유형"] = "스냅샷"

        # CSV에 append
        self.data_count += 1
        df = pd.DataFrame([self.last_data])
        df.to_csv(self.csv_path, mode="a", header=False, index=False, encoding="utf-8-sig")

        # 콘솔 출력
        print(f"[{self.data_count:4d}] {self.last_data['시간']} | "
              f"옵션: {self.last_data.get('옵션현재가', ''):>8} | "
              f"IV: {self.last_data.get('옵션IV', ''):>8} | "
              f"델타: {self.last_data.get('옵션델타', ''):>7} | "
              f"지수: {self.last_data.get('지수현재가', ''):>10}")

    def run(self):
        """메인 실행"""
        if not self.option_code:
            print("[오류] OPTION_CODE가 설정되지 않았습니다.")
            return

        self.login()

        if not self.connected:
            print("로그인 실패로 종료합니다.")
            return

        print("\n" + "=" * 60)
        print(f"KOSPI200 지수옵션 실시간 호가 수집 시작")
        print(f"  - 옵션: {self.option_code}")
        print(f"  - 지수: {self.index_code} (KOSPI200)")
        print(f"  - 저장 간격: {self.interval_ms}ms")
        print("=" * 60)

        self.set_real_reg()

        # 타이머 시작
        self.timer.start(self.interval_ms)

        print(f"\n실시간 데이터 수집 중... (중지: Kernel Interrupt)")
        print(f"저장 위치: {self.csv_path}")
        print(f"저장 방식: {self.interval_ms/1000:.1f}초마다 스냅샷 저장\n")

        self.app.exec_()

    def stop(self):
        """수집 중지"""
        self.timer.stop()
        print(f"\n수집 종료. 총 {self.data_count}개 데이터 저장됨.")
        print(f"저장 파일: {self.csv_path}")
        print(f"수신된 타입 목록: {self.received_types}")
        self.app.quit()

## 5. 실행

In [ ]:
# 수집기 생성 및 실행
collector = KiwoomIndexOptionCollector(
    option_code=OPTION_CODE,
    index_code=INDEX_CODE,
    screen_no=SCREEN_NO,
    csv_path=CSV_PATH,
    interval_ms=1000  # 1초마다 스냅샷 저장
)

# 실행 (중지하려면 Kernel > Interrupt)
collector.run()

CSV 파일 생성: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\option,futures MM\data\kospi200_option_B0164850_20260317_1414.csv

[성공] 로그인 완료
사용자: 박상혁
서버: 실투자

KOSPI200 지수옵션 실시간 호가 수집 시작
  - 옵션: B0164850
  - 지수: 201 (KOSPI200)
  - 저장 간격: 1000ms
옵션 실시간 등록 (B0164850): 0
지수 실시간 등록 (201): 0

실시간 데이터 수집 중... (중지: Kernel Interrupt)
저장 위치: c:\Users\adg01\OneDrive\바탕 화면\Yonsei\26-1 Yonsei\Y-FoRM\Y-FoRM 장플\option,futures MM\data\kospi200_option_B0164850_20260317_1414.csv
저장 방식: 1.0초마다 스냅샷 저장


>>> [NEW TYPE] real_type = '옵션이론가' (code: B0164850) <<<

>>> [NEW TYPE] real_type = '업종등락' (code: 201) <<<

>>> [NEW TYPE] real_type = '업종지수' (code: 201) <<<
[   1] 2026-03-17 14:15:19 | 옵션:          | IV:  52.8756 | 델타:  0.3958 | 지수:     851.94

>>> [NEW TYPE] real_type = '옵션호가잔량' (code: B0164850) <<<
[   2] 2026-03-17 14:15:20 | 옵션:          | IV:  52.8756 | 델타:  0.3958 | 지수:     851.94
[   3] 2026-03-17 14:15:21 | 옵션:          | IV:  52.8756 | 델타:  0.3958 | 지수:     851.94
[   4] 2026-03-1

KeyboardInterrupt: 

[ 394] 2026-03-17 14:21:54 | 옵션:    39.45 | IV:  52.9745 | 델타:  0.3956 | 지수:     852.16
[ 395] 2026-03-17 14:21:55 | 옵션:    39.45 | IV:  53.0339 | 델타:  0.3955 | 지수:     852.34
[ 396] 2026-03-17 14:21:56 | 옵션:    39.45 | IV:  53.0220 | 델타:  0.3955 | 지수:     852.31
[ 397] 2026-03-17 14:21:57 | 옵션:    39.45 | IV:  53.1050 | 델타:  0.3954 | 지수:     852.52
[ 398] 2026-03-17 14:21:58 | 옵션:    39.45 | IV:  52.9864 | 델타:  0.3956 | 지수:     852.22
[ 399] 2026-03-17 14:21:59 | 옵션:    39.55 | IV:  53.1209 | 델타:  0.3954 | 지수:     852.56
[ 400] 2026-03-17 14:22:00 | 옵션:    39.55 | IV:  53.1248 | 델타:  0.3953 | 지수:     852.57
[ 401] 2026-03-17 14:22:01 | 옵션:    39.55 | IV:  53.1564 | 델타:  0.3953 | 지수:     852.65
[ 402] 2026-03-17 14:22:02 | 옵션:    39.55 | IV:  53.2315 | 델타:  0.3951 | 지수:     852.84
[ 403] 2026-03-17 14:22:03 | 옵션:    39.55 | IV:  53.2434 | 델타:  0.3951 | 지수:     852.87
[ 404] 2026-03-17 14:22:04 | 옵션:    39.55 | IV:  53.1564 | 델타:  0.3953 | 지수:     852.65


: 

## 6. 수집된 데이터 확인

In [ ]:
import pandas as pd

# 데이터 로드
try:
    df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
except:
    df = pd.read_csv(r"data/kospi200_option_B0164812_YYYYMMDD_HHMM.csv", encoding="utf-8-sig")

print(f"총 행 수: {len(df)}")
print(f"수집 시간 범위: {df['시간'].iloc[0]} ~ {df['시간'].iloc[-1]}")
print(f"\n=== 컬럼별 데이터 유무 ===")
for col in df.columns:
    non_null = df[col].notna().sum()
    non_empty = (df[col] != "").sum() if df[col].dtype == object else non_null
    print(f"{col}: {non_empty}/{len(df)} ({non_empty/len(df)*100:.1f}%)")

In [ ]:
# 옵션 데이터 확인
print("=== 옵션 데이터 (최근 10건) ===")
df[["시간", "옵션현재가", "옵션IV", "옵션델타", "옵션감마", 
    "옵션세타", "옵션베가", "옵션매수호가1", "옵션매도호가1"]].tail(10)

In [ ]:
# KOSPI200 지수 데이터 확인
print("=== KOSPI200 지수 데이터 (최근 10건) ===")
df[["시간", "지수현재가", "지수전일대비", "지수등락률", 
    "지수시가", "지수고가", "지수저가"]].tail(10)